## RAG

#### 全局配置

In [ ]:
MILVUS_URI = "http://localhost:19530"
DB_NAME = "rag_test"
COLLECTION_NAME = "docs"
KNOWLEDGE_FILE = "./assets/knowledge.txt"

MODEL = "deepseek:deepseek-flash"
EMBED_MODEL = "text-embedding-v4"
EMBED_DIM = 1024

#### 初始化

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient(MILVUS_URI)

# 初始化数据库
existed_databases = client.list_databases()
if DB_NAME not in existed_databases:
    client.create_database(DB_NAME)
client.use_database(DB_NAME)

# 初始化集合
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBED_DIM,
    metric_type="COSINE",
)

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv(override=True)

embed_model = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_API_BASE"),
    check_embedding_ctx_length=False,
    chunk_size=10,
)

#### 文档读取并切分

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader(file_path=KNOWLEDGE_FILE, encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=["\n\n", "\n", "。", " ", ""]
)
chunks = splitter.split_documents(documents)
print(len(chunks))

for i, chunk in enumerate(chunks):
    print(i, chunk)
    print(f"\nchunk{i}: ", chunk.page_content)

#### 生成向量并落库

In [ ]:
texts = [
    chunk.page_content for chunk in chunks
]
vectors = embed_model.embed_documents(texts)

In [ ]:
data = [
    {
        "id": i,
        "vector": vectors[i],
        "text": texts[i],
        "source": chunks[i].metadata['source'],
    } for i in range(len(vectors))
]
client.upsert(COLLECTION_NAME, data)
client.flush(COLLECTION_NAME)

client.get_collection_stats(COLLECTION_NAME)

#### 创建 Agent

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str):
    """搜索知识库"""
    embed_query = embed_model.embed_query(query)
    search_res = client.search(
        collection_name=COLLECTION_NAME,
        data=[embed_query],
        limit=5,
        output_fields=["id", "text", "source"],
    )
    return "\n\n".join(
        f"[片段{i}]\n来源：{res["entity"].get('source', '未知')}\n"
        f"内容：{res["entity"]["text"]}"
        for i, res in enumerate(search_res[0], 1)
    )
    return results[0]

model = init_chat_model(
    model=MODEL,
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[search_knowledge_base],
    system_prompt=(
        "你是一个学习助手。"
        "请仅根据检索到的上下文回答问题。"
        "如果上下文不足以回答，请直接回答不知道。"
    )
)

#### 检索

In [ ]:
questions = [
    "RAG是什么？",
    "青禾知识助手支持哪些文件类型？",
    "青禾知识助手每月采购预算是多少？"
]
for question in questions:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("\n问题：", question)
    print("回答：", result["messages"][-1].content)
    print("-" * 60)